# 15 · Data-preparation diagnostics: angle scale (1P1Q), what constant padding repairs, hardware re-analysis

Added after the independent review of 11 Nov 2026. The questions are about **preparing jet data for a quantum circuit**; no new model is proposed.

### A. Angle scale in 1P1Q
For small rotation angles the fidelity kernel is, to leading order, $K(a, a') \approx 1 - \tfrac14 \sum_i (a_i - a'_i)^2$, where $a$ collects all angles $\theta_i, \phi_i$ of a jet.
In the SVM decision function $\sum_j \alpha_j y_j K(a_j, a) + b$ the $\lVert a\rVert^2$ term is multiplied by $\sum_j \alpha_j y_j = 0$ and drops out,
so the model is a **linear classifier in the angles**.
- Original 1P1Q: the angles are the signed $f z \Delta\eta$ and $f z \Delta\phi$. They are distributed symmetrically around 0 for quarks and for gluons, so a linear function of them carries almost no quark/gluon information.
- Rotation-invariant variant: the angles are $f z \Delta R \ge 0$; their linear sum is girth-like and discriminating.

**Prediction.** With exact kernels the higher-order (non-linear) terms can still be used at small $f$, but only with very large $C$, because the informative kernel differences are tiny.
With 1000 shots these differences are buried in shot noise, so at small $f$ the original 1P1Q falls to the level of a linear classifier on the signed angles (≈ 0.5), while the invariant variant stays close to a linear classifier on $z\,\Delta R$.
At finite shots the original 1P1Q works only when $f$ is large enough for the non-linear terms to be resolved.

### B. What does constant padding repair: the sign or the norm?
Plain amplitude encoding loses both: $x$ and $\lambda x$ give the same state for any $\lambda \neq 0$ (for $\lambda < 0$ up to a global phase).
Notebook 14 restored both at once. Here, on standardised inputs, two encodings restore **only one** of them:
- **norm kept, sign lost** (3 qubits): $|x/\lVert x\rVert\rangle \otimes R_Y(s\lVert x\rVert)|0\rangle$, kernel $\cos^2\angle(x, x') \cdot \cos^2\!\big(s(\lVert x\rVert - \lVert x'\rVert)/2\big)$;
- **sign kept, norm lost** (2 qubits): $(u, c)/\sqrt{1 + c^2}$ with $u = x/\lVert x\rVert$, kernel $(u\cdot u' + c^2)^2/(1 + c^2)^2$.

**Prediction.** If the sign is the cause, "sign kept, norm lost" recovers to the level of the c-padded encoding and "norm kept, sign lost" stays near the zero-padded one (≈ 0.62).

Also in B: the AUC of the c-padded kernel as a function of $k$ ($c = k \cdot \mathrm{median}\lVert x\rVert$), to see whether the recovery happens already at small $k$ or only where the kernel approaches an RBF kernel (large $k$); and a check that the two class means are exactly opposite after standardisation.

### C. Re-analysis of the saved IBM results (no new QPU time)
Rank correlation and linear fit for Experiment B; a consistency check of the Experiment A reference circuit.

### Protocol (fixed before the run)
- Training: development file `_MC_1`, balanced 400-jet subsamples, same event-level split as before. Seeds 0–4 (A) and 0–9 (B).
- Evaluation: `_MC_8`. It was used once in notebook 14, so it is **no longer untouched**; it is never used for any choice here.
- Every $C$ (and $s$, $c$, $f$ where they are tuned) is chosen by 5-fold CV inside the training subsample, scored with the **pT-reweighted AUC** (weights from the subsample itself, 6 log-spaced pT bins).
- Curves show the evaluation AUC at each fixed $f$ or $k$, with $C$ tuned by CV at that point. "CV-selected" rows also choose $f$, $k$, $s$ or $c$ by CV.
- Reported: mean ± std (ddof = 1) over seeds. Every SVM fit records whether it stopped at `MAX_ITER` (`fit_status_`); the counts are printed.
- Noisy (1000-shot) training kernels are passed to the SVM as they are (no projection onto positive semidefinite matrices), as in notebooks 11–14.

In [ ]:
import sys
sys.path.insert(0, "src")

import json
import time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import poisson, spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)   # not hidden: every capped fit is counted (fit_status_)

from jet_data import (load_jets, select_quark_gluon, feature_matrix, event_level_split, balanced_subsample,
                      leading_constituents, BOOKKEEPING, PF_KEYS)
from kernels import (preprocess, amplitude_kernel, amplitude_kernel_padded, angle_kernel,
                     onep1q_states, onep1q_kernel, shot_noise)
from figure_style import set_style, save, COLORS

set_style()
SEED = 42
SEEDS_A = [0, 1, 2, 3, 4]
SEEDS_B = list(range(10))
N_PER_CLASS = 200
N_CONST = 10
QGL_INPUTS = ("QG_mult", "QG_ptD", "QG_axis2")
F_GRID = np.unique(np.round(np.concatenate([np.logspace(0, 3, 10), [7.27]]), 2))   # 1 ... 1000, plus Bal et al.'s 7.27
C_GRID_A = np.logspace(-1, 7, 9)                  # part A: 0.1 ... 1e7 (the near-linear small-f regime needs very large C)
C_GRID_B = np.logspace(-2, 6, 9)                  # part B: 0.01 ... 1e6, as in notebook 14
K_GRID = [0.1, 0.25, 0.5, 1.0, 1.5, 2.0, 3.0, 5.0, 8.0, 13.0, 20.0]   # c-pad: c = k * median ||x||
S_K_GRID = [0.25, 0.5, 1.0, 2.0, 4.0]             # norm qubit: angle s ||x|| with s = 1 / (k * median ||x||)
SIGN_C_GRID = [0.25, 0.5, 1.0, 2.0, 4.0]          # sign-only encoding: (u, c) / sqrt(1 + c^2)
SHOTS = 1000
MAX_ITER = 200_000
DEV_PATH = Path("Data/JetNtuple_RunIISummer16_13TeV_MC_1.h5")
EVAL_PATH = Path("Data/JetNtuple_RunIISummer16_13TeV_MC_8.h5")
RESULTS = Path("results")
assert EVAL_PATH.exists(), f"evaluation file not found: {EVAL_PATH}"

def load(path):
    jets = select_quark_gluon(load_jets(path=path, features=QGL_INPUTS + ("jetPt",), extra=BOOKKEEPING + PF_KEYS),
                              require_loose_id=True)
    z, de, dp, _ = leading_constituents(jets, n_max=N_CONST)
    return {"jets": jets, "y": jets["label"], "X": feature_matrix(jets, features=QGL_INPUTS),
            "pt": np.asarray(jets["jetPt"], float), "z": z, "de": de, "dp": dp, "dr": np.sqrt(de**2 + dp**2)}

dev, ev = load(DEV_PATH), load(EVAL_PATH)
train, _ = event_level_split(dev["jets"], test_fraction=0.3, seed=SEED)
EV_ROWS = np.arange(len(ev["y"]))
print(f"dev {len(dev['y'])} jets; evaluation file {len(ev['y'])} jets, quark fraction {ev['y'].mean():.3f}")

### Helpers
- `pt_weights`: gluon jets get the weight $p_q(p_T)/p_g(p_T)$ (ratio of the normalised pT histograms), quark jets weight 1. With these weights pT alone has AUC 0.5.
- `tune_C`: 5-fold CV inside the 400 training jets, score = pT-reweighted AUC on the validation fold.
- `fit_svc`: every fit is counted, and counted as *capped* if libsvm stopped at `MAX_ITER`.

In [ ]:
def pt_weights(y, pt, n_bins=30, lo=30.0):
    edges = np.logspace(np.log10(lo), np.log10(pt.max() * 1.001), n_bins + 1)
    hq, _ = np.histogram(pt[y == 1], edges, density=True)
    hg, _ = np.histogram(pt[y == 0], edges, density=True)
    b = np.clip(np.digitize(pt, edges) - 1, 0, n_bins - 1)
    return np.where(y == 1, 1.0, np.divide(hq, hg, out=np.zeros_like(hq), where=hg > 0)[b])

W_EV = pt_weights(ev["y"], ev["pt"])
print(f"check: AUC of pT on the evaluation file, inclusive {roc_auc_score(ev['y'], ev['pt']):.3f}, "
      f"reweighted {roc_auc_score(ev['y'], ev['pt'], sample_weight=W_EV):.3f}")

FITS = {"total": 0, "capped": 0}
def fit_svc(K, y, C):
    svc = SVC(kernel="precomputed", C=C, max_iter=MAX_ITER).fit(K, y)
    FITS["total"] += 1
    FITS["capped"] += int(svc.fit_status_ != 0)
    return svc

def tune_C(K, y, w, folds, grid):
    best_score, best_C = -np.inf, None
    for C in grid:
        score = np.mean([roc_auc_score(y[va], fit_svc(K[np.ix_(tr, tr)], y[tr], C).decision_function(K[np.ix_(va, tr)]),
                                       sample_weight=w[va]) for tr, va in folds])
        if score > best_score:
            best_score, best_C = score, C
    return best_C, best_score

def evaluate(K_tr, K_ev, y_tr, C):
    scores = fit_svc(K_tr, y_tr, C).decision_function(K_ev)
    return roc_auc_score(ev["y"], scores, sample_weight=W_EV), roc_auc_score(ev["y"], scores)

def training_set(s):
    idx = balanced_subsample(dev["y"], train, N_PER_CLASS, seed=s)
    y_tr = dev["y"][idx]
    w_tr = pt_weights(y_tr, dev["pt"][idx], n_bins=6)
    folds = list(StratifiedKFold(5, shuffle=True, random_state=s).split(idx, y_tr))
    return idx, y_tr, w_tr, folds

def fits_since(start):
    return {k: FITS[k] - start[k] for k in FITS}

## A. Angle scale in 1P1Q (about 10–20 minutes)

For each $f$: kernels for the 400 training jets and for all evaluation jets, $C$ tuned by CV, evaluated once — with exact kernels and with 1000 shots (tuned and evaluated at the same budget).
Classical references (no $f$, since a linear model does not care about the overall scale): logistic regression on the signed angles $(z\Delta\eta, z\Delta\phi)$ and on $z\Delta R$, and the untrained sum $\sum_i z_i \Delta R_i$.

In [ ]:
def onep1q_inputs(D, rows, f, variant):
    if variant == "original":
        return onep1q_states(D["z"][rows], D["de"][rows], D["dp"][rows], f)
    return f * D["z"][rows] * D["dr"][rows]

def onep1q_k(P, Q, variant):
    return onep1q_kernel(P, Q) if variant == "original" else angle_kernel(P, Q)

VARIANTS_A = ("original", "invariant")
rec_A, start, t0 = [], dict(FITS), time.time()
for s in SEEDS_A:
    idx, y_tr, w_tr, folds = training_set(s)
    for v_i, variant in enumerate(VARIANTS_A):
        for f_i, f in enumerate(F_GRID):
            P_tr, P_ev = onep1q_inputs(dev, idx, f, variant), onep1q_inputs(ev, EV_ROWS, f, variant)
            K_tr, K_ev = onep1q_k(P_tr, P_tr, variant), onep1q_k(P_ev, P_tr, variant)
            for budget in (None, SHOTS):
                rng = np.random.default_rng([15, s, v_i, f_i, budget or 0])
                K_fit = shot_noise(K_tr, budget, rng, symmetric=True) if budget else K_tr
                K_e = shot_noise(K_ev, budget, rng) if budget else K_ev
                C, cv = tune_C(K_fit, y_tr, w_tr, folds, C_GRID_A)
                rw, inc = evaluate(K_fit, K_e, y_tr, C)
                rec_A.append({"variant": variant, "f": float(f), "budget": budget or "exact", "seed": s, "C": float(C),
                              "cv_auc_rw": float(cv), "auc_rw": float(rw), "auc_inc": float(inc),
                              "median_angle": float(np.median(np.abs(P_tr if variant == "invariant" else f * dev["z"][idx] * dev["de"][idx])))})
    print(f"seed {s} done ({time.time() - t0:.0f} s)", flush=True)
fits_A = fits_since(start)
print(f"SVM fits in part A: {fits_A['total']}, stopped at MAX_ITER: {fits_A['capped']} ({fits_A['capped'] / fits_A['total']:.1%})")

In [ ]:
def logistic_ref(F_tr, y_tr, w_tr, folds, F_ev):
    scaler = StandardScaler().fit(F_tr)
    A, E = scaler.transform(F_tr), scaler.transform(F_ev)
    best = max(np.logspace(-3, 3, 7), key=lambda C: np.mean([
        roc_auc_score(y_tr[va], LogisticRegression(C=C, max_iter=5000).fit(A[tr], y_tr[tr]).decision_function(A[va]), sample_weight=w_tr[va])
        for tr, va in folds]))
    return roc_auc_score(ev["y"], LogisticRegression(C=best, max_iter=5000).fit(A, y_tr).decision_function(E), sample_weight=W_EV)

ref_A = {"linear on signed angles (z dEta, z dPhi)": [], "linear on z dR": []}
for s in SEEDS_A:
    idx, y_tr, w_tr, folds = training_set(s)
    signed = lambda D, r: np.hstack([D["z"][r] * D["de"][r], D["z"][r] * D["dp"][r]])
    radial = lambda D, r: D["z"][r] * D["dr"][r]
    ref_A["linear on signed angles (z dEta, z dPhi)"].append(logistic_ref(signed(dev, idx), y_tr, w_tr, folds, signed(ev, EV_ROWS)))
    ref_A["linear on z dR"].append(logistic_ref(radial(dev, idx), y_tr, w_tr, folds, radial(ev, EV_ROWS)))
girth_like = -(ev["z"] * ev["dr"]).sum(axis=1)          # larger for gluons, so the quark score is minus the sum
ref_A["sum z dR (no training)"] = [roc_auc_score(ev["y"], girth_like, sample_weight=W_EV)]
for name, vals in ref_A.items():
    print(f"{name:45s} pT-reweighted AUC {np.mean(vals):.3f}" + (f" ± {np.std(vals, ddof=1):.3f}" if len(vals) > 1 else ""))

### A: results
Rows: AUC at each fixed $f$ (C tuned). Then the CV-selected $f$ per seed (both $f$ and $C$ chosen by CV).

In [ ]:
def rows(variant, budget, f=None):
    return [r for r in rec_A if r["variant"] == variant and r["budget"] == budget and (f is None or r["f"] == f)]

print(f"{'f':>8s}  {'original exact':>15s} {'original 1000':>15s}  {'invariant exact':>15s} {'invariant 1000':>15s}   median C (orig. exact / inv. exact)   median |angle| (orig. / inv.)")
for f in F_GRID:
    cells = []
    for variant in VARIANTS_A:
        for budget in ("exact", SHOTS):
            a = [r["auc_rw"] for r in rows(variant, budget, f)]
            cells.append(f"{np.mean(a):.3f} ± {np.std(a, ddof=1):.3f}")
    mc = [np.median([r["C"] for r in rows(v, "exact", f)]) for v in VARIANTS_A]
    ma = [np.median([r["median_angle"] for r in rows(v, "exact", f)]) for v in VARIANTS_A]
    print(f"{f:8.2f}  {cells[0]:>15s} {cells[1]:>15s}  {cells[2]:>15s} {cells[3]:>15s}   {mc[0]:9.0e} / {mc[1]:9.0e}            {ma[0]:.3g} / {ma[1]:.3g}")

print("\nCV-selected f (f and C chosen by CV inside the training subsample):")
selected_A = {}
for variant in VARIANTS_A:
    for budget in ("exact", SHOTS):
        best = [max([r for r in rows(variant, budget) if r["seed"] == s], key=lambda r: r["cv_auc_rw"]) for s in SEEDS_A]
        label = f"1P1Q {variant} ({'exact' if budget == 'exact' else f'{budget} shots'})"
        selected_A[label] = {"auc_rw": [r["auc_rw"] for r in best], "f": [r["f"] for r in best], "C": [r["C"] for r in best]}
        edge = sum(r["f"] in (F_GRID.min(), F_GRID.max()) for r in best)
        print(f"  {label:32s} {np.mean(selected_A[label]['auc_rw']):.3f} ± {np.std(selected_A[label]['auc_rw'], ddof=1):.3f};  "
              f"f = {[round(r['f'], 1) for r in best]}  (at grid edge in {edge}/{len(best)} seeds)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, budget, title in ((axes[0], "exact", "exact kernels"), (axes[1], SHOTS, f"{SHOTS} shots")):
    for variant, col, lab in (("original", COLORS["series1"], "1P1Q original (z Δη, z Δφ)"),
                              ("invariant", COLORS["series3"], "1P1Q rotation-invariant (z ΔR)")):
        m = np.array([np.mean([r["auc_rw"] for r in rows(variant, budget, f)]) for f in F_GRID])
        e = np.array([np.std([r["auc_rw"] for r in rows(variant, budget, f)], ddof=1) for f in F_GRID])
        ax.plot(F_GRID, m, "o-", color=col, label=lab, markersize=4)
        ax.fill_between(F_GRID, m - e, m + e, color=col, alpha=0.15, linewidth=0)
    ax.axhline(np.mean(ref_A["linear on signed angles (z dEta, z dPhi)"]), color=COLORS["series1"], ls=":", lw=1.2,
               label="classical linear, signed angles")
    ax.axhline(np.mean(ref_A["linear on z dR"]), color=COLORS["series3"], ls=":", lw=1.2, label="classical linear, z ΔR")
    ax.axvline(7.27, color=COLORS["reference"], ls="--", lw=1)
    ax.text(7.27 * 1.08, 0.505, "f of Bal et al.", color=COLORS["reference"], fontsize=8)
    ax.set_xscale("log"); ax.set_xlabel("scale factor f"); ax.set_title(title, loc="left", fontsize=10)
axes[0].set_ylabel("pT-reweighted AUC (MC_8)")
handles, labels = axes[0].get_legend_handles_labels()
fig.suptitle("1P1Q: AUC vs angle scale f (C tuned by CV at every f)", x=0.01, ha="left", fontsize=11)
fig.tight_layout(rect=(0, 0.1, 1, 1))
fig.legend(handles, labels, loc="lower center", ncol=4, fontsize=8, frameon=False)
save(fig, "fig16_onep1q_angle_scale")
plt.show()

## B. Sign or norm? (about 5–10 minutes)
Standardised inputs, exact kernels, 10 seeds. Each encoding's free parameter ($k$, $s$ or $c$) is chosen by CV; the c-padded kernel is also shown as a curve over $k$.

In [ ]:
rec_B, cos_means, start, t0 = [], [], dict(FITS), time.time()
for s in SEEDS_B:
    idx, y_tr, w_tr, folds = training_set(s)
    A, E = preprocess("standard", dev["X"][idx], ev["X"])
    mq, mg = A[y_tr == 1].mean(axis=0), A[y_tr == 0].mean(axis=0)
    cos_means.append(float(mq @ mg / (np.linalg.norm(mq) * np.linalg.norm(mg))))
    nA, nE = np.linalg.norm(A, axis=1), np.linalg.norm(E, axis=1)
    UA, UE = A / nA[:, None], E / nE[:, None]
    med = float(np.median(nA))

    def run(name, param, K_tr, K_ev):
        C, cv = tune_C(K_tr, y_tr, w_tr, folds, C_GRID_B)
        rw, inc = evaluate(K_tr, K_ev, y_tr, C)
        rec_B.append({"variant": name, "param": param, "seed": s, "C": float(C), "cv_auc_rw": float(cv),
                      "auc_rw": float(rw), "auc_inc": float(inc)})

    run("zero-pad (sign and norm lost)", None, amplitude_kernel(A, A), amplitude_kernel(E, A))
    for k in K_GRID:
        run("c-pad (sign and norm kept)", k, amplitude_kernel_padded(A, A, k * med), amplitude_kernel_padded(E, A, k * med))
    for k in S_K_GRID:
        sp = 1.0 / (k * med)
        norm_k = lambda P, Q, nP, nQ: amplitude_kernel(P, Q) * np.cos(sp * (nP[:, None] - nQ[None, :]) / 2) ** 2
        run("norm kept, sign lost", k, norm_k(A, A, nA, nA), norm_k(E, A, nE, nA))
    for c in SIGN_C_GRID:
        run("sign kept, norm lost", c, amplitude_kernel_padded(UA, UA, c), amplitude_kernel_padded(UE, UA, c))
    Am, Em = preprocess("minmax", dev["X"][idx], ev["X"])
    run("min-max, zero-pad (reference)", None, amplitude_kernel(Am, Am), amplitude_kernel(Em, Am))
    print(f"seed {s} done ({time.time() - t0:.0f} s)", flush=True)
fits_B = fits_since(start)
print(f"SVM fits in part B: {fits_B['total']}, stopped at MAX_ITER: {fits_B['capped']} ({fits_B['capped'] / fits_B['total']:.1%})")
print(f"cosine between the standardised quark and gluon class means: {np.round(cos_means, 6).tolist()}")

In [ ]:
VARIANTS_B = ["zero-pad (sign and norm lost)", "norm kept, sign lost", "sign kept, norm lost",
              "c-pad (sign and norm kept)", "min-max, zero-pad (reference)"]
selected_B = {}
print(f"{'encoding (standardised inputs unless stated)':42s} {'pT-reweighted AUC':>18s}   CV-chosen parameter (per seed)")
for name in VARIANTS_B:
    best = [max([r for r in rec_B if r["variant"] == name and r["seed"] == s], key=lambda r: r["cv_auc_rw"]) for s in SEEDS_B]
    selected_B[name] = {"auc_rw": [r["auc_rw"] for r in best], "param": [r["param"] for r in best]}
    a = selected_B[name]["auc_rw"]
    print(f"{name:42s} {np.mean(a):8.3f} ± {np.std(a, ddof=1):.3f}   {[r['param'] for r in best]}")

def paired(a, b):
    d = np.array(selected_B[a]["auc_rw"]) - np.array(selected_B[b]["auc_rw"])
    print(f"  {a} − {b}: {d.mean():+.3f} (SE {d.std(ddof=1) / np.sqrt(len(d)):.3f}), positive in {np.sum(d > 0)}/{len(d)} seeds")
print("\nPaired over seeds:")
paired("sign kept, norm lost", "zero-pad (sign and norm lost)")
paired("norm kept, sign lost", "zero-pad (sign and norm lost)")
paired("c-pad (sign and norm kept)", "sign kept, norm lost")

print("\nc-pad, AUC at fixed k (C tuned):")
curve_B = {}
for k in K_GRID:
    a = [r["auc_rw"] for r in rec_B if r["variant"] == "c-pad (sign and norm kept)" and r["param"] == k]
    curve_B[k] = a
    print(f"  k = {k:5.2f}: {np.mean(a):.3f} ± {np.std(a, ddof=1):.3f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.0), gridspec_kw={"width_ratios": [1.1, 1]})
ypos = np.arange(len(VARIANTS_B))[::-1]
cols = [COLORS["series1"], COLORS["series1"], COLORS["series3"], COLORS["series3"], COLORS["reference"]]
for yp, name, col in zip(ypos, VARIANTS_B, cols):
    a = selected_B[name]["auc_rw"]
    ax1.errorbar(np.mean(a), yp, xerr=np.std(a, ddof=1), fmt="o", color=col, markersize=8, capsize=3, markeredgecolor="white")
ax1.set_yticks(ypos, VARIANTS_B)
ax1.set_xlabel("pT-reweighted AUC (MC_8)")
ax1.set_title("Standardised inputs: which lost property matters, sign or norm?", loc="left", fontsize=10)
ax1.grid(axis="y", visible=False)

m = np.array([np.mean(curve_B[k]) for k in K_GRID]); e = np.array([np.std(curve_B[k], ddof=1) for k in K_GRID])
ax2.plot(K_GRID, m, "o-", color=COLORS["series3"], markersize=4, label="c-pad (standardised)")
ax2.fill_between(K_GRID, m - e, m + e, color=COLORS["series3"], alpha=0.15, linewidth=0)
ax2.axhline(np.mean(selected_B["zero-pad (sign and norm lost)"]["auc_rw"]), color=COLORS["series1"], ls=":", lw=1.2, label="zero-pad (standardised)")
ax2.axhline(np.mean(selected_B["min-max, zero-pad (reference)"]["auc_rw"]), color=COLORS["reference"], ls=":", lw=1.2, label="min-max, zero-pad")
ax2.set_xscale("log"); ax2.set_xlabel("k   (c = k · median ||x||)")
ax2.set_xticks([0.1, 0.3, 1, 3, 10], ["0.1", "0.3", "1", "3", "10"]); ax2.minorticks_off()
ax2.set_title("c-pad: AUC vs padding constant (C tuned at every k)", loc="left", fontsize=10)
ax2.legend(fontsize=8, loc="center right")
fig.tight_layout()
save(fig, "fig17_sign_or_norm")
plt.show()

## C. Re-analysis of the saved IBM results (Experiment A and B, `results/ibm_results.json`)
- **B:** Pearson and Spearman correlation, linear fit (slope, intercept) and the mean offset between measured and exact kernel values, with the number of distinct jets behind the 28 pairs.
- **A:** if the empty reference circuit had run on the same physical qubits as the jet circuits, every jet circuit would show at least about as many wrong "10" outcomes as the reference. The Poisson probability of the observed count is printed. (The circuits were transpiled one by one without a fixed `initial_layout`, so the qubits can differ.)

In [ ]:
hw = json.load(open(RESULTS / "ibm_results.json"))
meta, counts, shots = hw["meta"], hw["counts"], hw["run"]["shots"]
prob = lambda c, b: c.get(b, 0) / shots
hw_B = {}
for prep in ("raw", "minmax"):
    sel = [(m, c) for m, c in zip(meta, counts) if m["exp"] == "B" and m["prep"] == prep]
    ex = np.array([m["exact"] for m, _ in sel]); me = np.array([prob(c, "00") for _, c in sel])
    slope, intercept = np.polyfit(ex, me, 1)
    hw_B[prep] = {"pearson": float(np.corrcoef(ex, me)[0, 1]), "spearman": float(spearmanr(ex, me)[0]),
                  "slope": float(slope), "intercept": float(intercept), "mean_offset": float(np.mean(ex - me)),
                  "exact_range": [float(ex.min()), float(ex.max())], "n_pairs": len(sel),
                  "n_jets": len({j for m, _ in sel for j in m["pair"]})}
    h = hw_B[prep]
    print(f"B, {prep:6s}: {h['n_pairs']} pairs of {h['n_jets']} jets; Pearson {h['pearson']:.3f}, Spearman {h['spearman']:.3f}; "
          f"fit measured = {h['slope']:.3f} · exact + {h['intercept']:.3f}; mean offset (exact − measured) {h['mean_offset']:.4f}; "
          f"exact range width {ex.max() - ex.min():.4f}")

ref = counts[0]
p_ref = prob(ref, "10")
print(f"\nA: reference circuit (ideal: always 00): '10' in {ref.get('10', 0)} / {shots} shots")
hw_A = []
for m, c in zip(meta, counts):
    if m["exp"] == "A" and m["kind"] == "jet":
        lam = shots * (m["exact"][2] + p_ref)          # expected '10' counts if the reference error applied to this circuit
        k = c.get("10", 0)
        hw_A.append({"jet": m["jet"], "count_10": k, "expected_if_same_qubits": lam, "poisson_p": float(poisson.cdf(k, lam))})
        print(f"   jet {m['jet']:6d}: '10' in {k:3d} shots; expected ≥ {lam:5.1f} if same qubits; P(≤ observed) = {poisson.cdf(k, lam):.1e}")

## D. Save and summary for the research log

In [ ]:
json.dump({"A": {"records": rec_A, "references": ref_A, "cv_selected": selected_A, "fits": fits_A},
           "B": {"records": rec_B, "cv_selected": selected_B, "class_mean_cosine": cos_means, "fits": fits_B},
           "C": {"B": hw_B, "A": hw_A}},
          open(RESULTS / "data_prep_diagnostics.json", "w"), indent=2)
print("saved results/data_prep_diagnostics.json\n")

print(f"- Data-preparation diagnostics (nb 15; train MC_1 400 jets, evaluate MC_8, C tuned by pT-reweighted CV, std ddof=1):")
print(f"  - A, 1P1Q angle scale ({len(SEEDS_A)} seeds): at f = 7.27 original exact "
      f"{np.mean([r['auc_rw'] for r in rows('original', 'exact', 7.27)]):.3f}, 1000 shots {np.mean([r['auc_rw'] for r in rows('original', SHOTS, 7.27)]):.3f}; "
      f"invariant exact {np.mean([r['auc_rw'] for r in rows('invariant', 'exact', 7.27)]):.3f}, 1000 shots {np.mean([r['auc_rw'] for r in rows('invariant', SHOTS, 7.27)]):.3f}")
for name, vals in ref_A.items():
    print(f"    reference {name}: {np.mean(vals):.3f}")
for label, d in selected_A.items():
    print(f"    CV-selected {label}: {np.mean(d['auc_rw']):.3f} ± {np.std(d['auc_rw'], ddof=1):.3f}, f {d['f']}")
print(f"    SVM fits {fits_A['total']}, capped at MAX_ITER {fits_A['capped']}")
print(f"  - B, sign or norm ({len(SEEDS_B)} seeds, exact): class-mean cosine {np.mean(cos_means):.6f}")
for name in VARIANTS_B:
    print(f"    {name}: {np.mean(selected_B[name]['auc_rw']):.3f} ± {np.std(selected_B[name]['auc_rw'], ddof=1):.3f}")
print(f"    c-pad vs k: " + ", ".join(f"k {k:g} {np.mean(curve_B[k]):.3f}" for k in K_GRID))
print(f"    SVM fits {fits_B['total']}, capped at MAX_ITER {fits_B['capped']}")
for prep, h in hw_B.items():
    print(f"  - C, hardware B {prep}: Pearson {h['pearson']:.3f}, Spearman {h['spearman']:.3f}, slope {h['slope']:.3f}, intercept {h['intercept']:.3f}, "
          f"offset {h['mean_offset']:.4f}, {h['n_jets']} jets")
print(f"  - C, hardware A: reference '10' rate {p_ref:.4f}; jet circuits '10' counts {[a['count_10'] for a in hw_A]}, "
      f"Poisson P(≤ obs | same qubits) max {max(a['poisson_p'] for a in hw_A):.1e}")